# Shot Data #346 : mesurer ses progrès avec le top-k accuracy
L'accuracy ne bouge pas, mais le modèle progresse : le top-k accuracy le révèle.

In [1]:
import numpy as np
from sklearn.metrics import accuracy_score, top_k_accuracy_score

classes = ["A", "B", "C", "D"]
y_true = np.array([2])  # vrai label : Classe C (indice 2)

# Probabilités prédites, dans l'ordre des classes A, B, C, D
v1 = np.array([[0.20, 0.60, 0.05, 0.15]])  # Version 1 : Classe C en dernière position
v2 = np.array([[0.15, 0.55, 0.25, 0.05]])  # Version 2 : Classe C en deuxième position

In [2]:
labels = np.arange(len(classes))
for nom, proba in [("Version 1", v1), ("Version 2", v2)]:
    pred = proba.argmax(axis=1)
    acc = accuracy_score(y_true, pred)
    top2 = top_k_accuracy_score(y_true, proba, k=2, labels=labels)
    top3 = top_k_accuracy_score(y_true, proba, k=3, labels=labels)
    print(f"{nom} | prédiction : Classe {classes[pred[0]]} | accuracy = {acc:.0f} | top-2 = {top2:.0f} | top-3 = {top3:.0f}")

Version 1 | prédiction : Classe B | accuracy = 0 | top-2 = 0 | top-3 = 0
Version 2 | prédiction : Classe B | accuracy = 0 | top-2 = 1 | top-3 = 1


Les deux versions prédisent `Classe B` : même accuracy (0). Mais la `Classe C` est passée de la dernière à la deuxième position : le top-2 et le top-3 le voient.

Attention : avec 4 classes, `k=4` donnerait toujours 1 (scikit-learn le signale par un avertissement quand k ≥ nombre de classes).

## Sur plusieurs itérations
Petite simulation : au fil des itérations, la vraie classe monte dans le classement, le top-3 accuracy prend de l'avance sur l'accuracy dès les premières itérations.

In [3]:
rng = np.random.default_rng(42)
n, n_classes = 1000, 10
y = rng.integers(0, n_classes, n)

def simuler(force):
    # Scores aléatoires, avec un bonus de plus en plus fort sur la vraie classe
    scores = rng.normal(size=(n, n_classes))
    scores[np.arange(n), y] += force
    exp = np.exp(scores)
    return exp / exp.sum(axis=1, keepdims=True)

for it, force in enumerate([0.5, 1.0, 1.5, 2.0, 2.5, 3.0], start=1):
    proba = simuler(force)
    acc = accuracy_score(y, proba.argmax(axis=1))
    top3 = top_k_accuracy_score(y, proba, k=3, labels=np.arange(n_classes))
    print(f"Itération {it} | accuracy = {acc:.2f} | top-3 accuracy = {top3:.2f}")

Itération 1 | accuracy = 0.20 | top-3 accuracy = 0.46
Itération 2 | accuracy = 0.36 | top-3 accuracy = 0.65
Itération 3 | accuracy = 0.51 | top-3 accuracy = 0.80
Itération 4 | accuracy = 0.69 | top-3 accuracy = 0.91
Itération 5 | accuracy = 0.79 | top-3 accuracy = 0.96
Itération 6 | accuracy = 0.89 | top-3 accuracy = 0.98
